# 01 — Data cleaning & target construction

**What Makes a Song Sticky?** This notebook loads Spotify track-level audio features and prepares analysis-ready tables.

**Framing:** We are *not* measuring addiction, replays, skips, or saves. **Popularity** is a public **proxy** for broad replayability. **Stickiness** here means: tracks in the **top ~20%** of popularity in *this* dataset (at or above the 80th percentile).

**Working directory:** Run Jupyter from the repo root, or from `notebooks/` — the next cell resolves `PROJECT_ROOT` automatically.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import data_prep

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "spotify_tracks.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CLEAN_PATH = PROCESSED_DIR / "spotify_cleaned.csv"
MODEL_PATH = PROCESSED_DIR / "spotify_model_data.csv"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


Project root: /Users/dhruvsood/SongAddiction


## 1. Load raw data


In [2]:
df_raw = data_prep.load_data(RAW_PATH)


## 2. Inspect: columns, head, info, missingness


In [3]:
print("Column names:", list(df_raw.columns))
print("Shape:", df_raw.shape)
display(df_raw.head())
df_raw.info()
miss = pd.DataFrame({
    "missing": df_raw.isna().sum(),
    "pct": (df_raw.isna().mean() * 100).round(2),
})
display(miss[miss["missing"] > 0] if miss["missing"].sum() else miss.head(0))
print("Missing summary (all columns):")
display(miss)


Column names: ['track_name', 'artist_name', 'genre', 'popularity', 'danceability', 'energy', 'valence', 'tempo', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'duration_ms']
Shape: (2500, 14)


,track_name,artist_name,genre,popularity,danceability,energy,valence,tempo,loudness,speechiness,acousticness,instrumentalness,liveness,duration_ms
0,track_0,artist_0,hip hop,47,0.643694,0.374511,0.295054,72.536867,-11.078831,0.054592,0.552008,0.066125,0.002421,277720
1,track_1,artist_1,hip hop,43,0.610431,0.530947,0.407379,91.558115,-19.231136,0.072337,0.604792,0.083395,0.592764,175640
2,track_2,artist_2,pop,39,0.536584,0.327675,0.653149,131.740492,-6.893558,0.166540,0.319003,0.113229,0.218477,331554
3,track_3,artist_3,electronic,31,0.396149,0.336000,0.636129,123.331297,-8.228587,0.079081,0.377835,0.011935,0.350695,399367
4,track_4,artist_4,rock,36,0.831866,0.166239,0.303035,90.517036,-19.306823,0.303159,0.551371,0.011418,0.021601,275456


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   track_name        2500 non-null   object 
 1   artist_name       2500 non-null   object 
 2   genre             2500 non-null   object 
 3   popularity        2500 non-null   int64  
 4   danceability      2500 non-null   float64
 5   energy            2500 non-null   float64
 6   valence           2500 non-null   float64
 7   tempo             2500 non-null   float64
 8   loudness          2500 non-null   float64
 9   speechiness       2500 non-null   float64
 10  acousticness      2500 non-null   float64
 11  instrumentalness  2500 non-null   float64
 12  liveness          2500 non-null   float64
 13  duration_ms       2500 non-null   int64  
dtypes: float64(9), int64(2), object(3)
memory usage: 273.6+ KB


,missing,pct


Missing summary (all columns):


,missing,pct
track_name,0,0.0
artist_name,0,0.0
genre,0,0.0
popularity,0,0.0
danceability,0,0.0
energy,0,0.0
valence,0,0.0
tempo,0,0.0
loudness,0,0.0
speechiness,0,0.0


## 3. Cleaning pipeline (automated)

The module [`src/data_prep.py`](../src/data_prep.py) applies, in order:

1. **Column mapping** — `COLUMN_MAP` maps common alternate names (e.g. `artists` → `artist_name`, `duration` → `duration_ms`) to canonical names.
2. **Duration** — `duration_sec` or values that look like **seconds** are converted to milliseconds when needed.
3. **Duplicates** — dropped on `track_name` + `artist_name` + `duration_ms` when those columns exist.
4. **Missing values** — rows with missing **popularity** or any core **audio feature** are dropped (optional fields like `genre` are not required).
5. **Validation** — popularity in [0, 100]; Spotify features in [0, 1]; `duration_ms` > 0.
6. **Targets** — `popularity_z` (z-score); `sticky` = 1 if popularity ≥ 80th percentile (top ~20%), else 0.


In [4]:
df, stats = data_prep.clean_dataframe(df_raw)
print("Sticky threshold (80th percentile):", stats.get("sticky_threshold"))
print("Class balance (proportion sticky):", round(stats.get("sticky_balance", 0), 4))
print("Rows after cleaning:", stats.get("n_rows_final"))
print("Duplicates removed:", stats.get("duplicates_removed"))
print("Validation notes:", stats.get("validation_notes"))


Sticky threshold (80th percentile): 56.0
Class balance (proportion sticky): 0.2072
Rows after cleaning: 2500
Duplicates removed: 0
Validation notes: {}


## 4. Summary statistics (cleaned numeric columns)


In [5]:
num_cols = df.select_dtypes(include=[np.number]).columns
display(df[num_cols].describe().T)


,count,mean,std,min,25%,50%,75%,max
popularity,2500.0,4.095040e+01,17.452317,5.000000,27.000000,39.000000,53.000000,100.000000
danceability,2500.0,5.060826e-01,0.221909,0.013338,0.334498,0.506138,0.677879,0.987340
energy,2500.0,4.951595e-01,0.224103,0.013560,0.325211,0.496846,0.664728,0.996277
valence,2500.0,4.980822e-01,0.221776,0.017985,0.325972,0.494926,0.670783,0.997206
tempo,2500.0,1.246286e+02,31.461537,70.014301,97.510951,124.781100,152.043679,179.987541
loudness,2500.0,-1.149413e+01,4.917347,-19.984634,-15.844649,-11.413517,-7.062424,-3.000732
speechiness,2500.0,1.686645e-01,0.142359,0.000029,0.057680,0.127676,0.251053,0.773205
acousticness,2500.0,3.987032e-01,0.197453,0.007787,0.243892,0.386267,0.535376,0.976320
instrumentalness,2500.0,1.075999e-01,0.097504,0.000040,0.034648,0.080247,0.152203,0.584572
liveness,2500.0,2.015185e-01,0.162341,0.000106,0.072408,0.163645,0.294949,0.895917


## 5. Save processed datasets


In [6]:
keep_meta = [c for c in ("track_name", "artist_name", "genre") if c in df.columns]
keep_num = [c for c in data_prep.EXPECTED_COLUMNS if c in df.columns and c not in keep_meta]
extra = ["popularity_z", "sticky"]
all_cols = keep_meta + [c for c in keep_num if c not in keep_meta] + [e for e in extra if e in df.columns]
df_out = df[all_cols].copy()

data_prep.save_processed_data(df_out, CLEAN_PATH)
data_prep.save_processed_data(df_out, MODEL_PATH)
print("Saved:", CLEAN_PATH)
print("Saved:", MODEL_PATH)


Saved: /Users/dhruvsood/SongAddiction/data/processed/spotify_cleaned.csv
Saved: /Users/dhruvsood/SongAddiction/data/processed/spotify_model_data.csv


## 6. Next steps

Run **`02_eda.ipynb`** on `spotify_model_data.csv`, then **`03_modeling.ipynb`**.
